# 🌿 Darukaa Reference Benchmarking Pipeline

**Compare any biodiversity indicator against ecoregion-specific reference benchmarks.**

This notebook runs the full pipeline:
1. Clone the repo & install dependencies
2. Authenticate Google Earth Engine
3. Upload your KML site file
4. Run Tier 1 (global) + Tier 2 (contemporary) benchmarking
5. Download the scorecard (JSON + CSV)

**Methodology:**
- McElderry et al. (2024) DOI:10.32942/X2689N — SEED reference selection
- McNellie et al. (2020) DOI:10.1111/gcb.15383 — Contemporary reference states
- Yen et al. (2019) DOI:10.1002/eap.1970 — Biodiversity benchmarks

---

## 1. Setup — Clone repo & install

In [ ]:
# Clone the repo (change URL to your private repo)
# For private repos: use a GitHub Personal Access Token
# !git clone https://<TOKEN>@github.com/darukaa-earth/reference-benchmarking.git

# For public repos:
!git clone https://github.com/darukaa-earth/reference-benchmarking.git

# Move into the repo
%cd reference-benchmarking

# Install the package
!pip install -e . -q

## 2. Authenticate Google Earth Engine

Run the cell below — it will show a link. Click it, sign in with your Google account, copy the token back.

In [ ]:
import ee

# Colab has a built-in GEE authenticator
ee.Authenticate()

# Initialize with your GEE project
# Replace with your actual project ID
GEE_PROJECT = "your-gee-project-id"  # @param {type:"string"}

ee.Initialize(project=GEE_PROJECT)
print(f"✓ GEE initialised with project: {GEE_PROJECT}")

## 3. Upload your KML file

Run the cell below to upload your project site KML/KMZ/GeoJSON file.

In [ ]:
from google.colab import files

print("Upload your KML/KMZ/GeoJSON file:")
uploaded = files.upload()

# Get the filename
SITE_FILE = list(uploaded.keys())[0]
print(f"\n✓ Uploaded: {SITE_FILE} ({len(uploaded[SITE_FILE])/1024:.1f} KB)")

## 4. Configure the pipeline

Adjust parameters below as needed.

In [ ]:
# ── Configuration ─────────────────────────────────────────

# Which indicators to run?
# Options: "ndvi", "lst", "bii", "eii", "ghm", "msa_globio4", "seed"
# Leave empty for ALL GEE-based indicators
INDICATORS = ["ndvi", "bii", "eii", "ghm", "lst"]  # @param

# Tier 2 reference selection
BUFFER_KM = 100        # Search radius for reference patches (km)
HMI_PERCENTILE = 5     # Top N% least disturbed = reference

# NDVI year
NDVI_YEAR = 2024       # @param {type:"integer"}

# ─────────────────────────────────────────────────────────
print(f"Indicators: {INDICATORS}")
print(f"Tier 2: buffer={BUFFER_KM}km, HMI top {HMI_PERCENTILE}%")
print(f"NDVI year: {NDVI_YEAR}")

## 5. Run the pipeline

In [ ]:
import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)

from darukaa_reference.config import Config
from darukaa_reference.registry import IndicatorRegistry
from darukaa_reference.indicators import create_default_registry
from darukaa_reference.pipeline import Pipeline

# Build config programmatically (no YAML needed in Colab)
config = Config(
    gee_project=GEE_PROJECT,
    ecoregion_source="gee",
    reference_buffer_km=BUFFER_KM,
    hmi_percentile_threshold=HMI_PERCENTILE,
    ndvi_year=NDVI_YEAR,
    lst_year=NDVI_YEAR,
    output_dir="./output",
    output_format="both",
    enabled_indicators=INDICATORS,
)

# Create registry
registry = create_default_registry()

# Build and run pipeline
pipeline = Pipeline(config, registry)
report = pipeline.run(
    site_path=SITE_FILE,
    output_path="./output/benchmark_scorecard",
)

## 6. View results

In [ ]:
import pandas as pd

# Load scorecard as DataFrame
df = pd.DataFrame(report["scorecard"])

# Display key columns
display_cols = [
    "site_id", "display_name", "pillar", "site_value",
    "tier1_reference", "tier1_intactness",
    "tier2_reference", "tier2_intactness",
    "hedges_g", "permutation_p", "interpretation",
]
existing_cols = [c for c in display_cols if c in df.columns]

print(f"\n{'='*70}")
print(f"BENCHMARK SCORECARD: {len(df)} indicator-site pairs")
print(f"{'='*70}\n")

df[existing_cols].style.format({
    "site_value": "{:.4f}",
    "tier1_reference": "{:.4f}",
    "tier1_intactness": "{:.1%}",
    "tier2_reference": "{:.4f}",
    "tier2_intactness": "{:.1%}",
    "hedges_g": "{:.3f}",
    "permutation_p": "{:.4f}",
}, na_rep="—")

In [ ]:
# Pillar summary
if report.get("pillar_summary"):
    print("\nPILLAR INTACTNESS SUMMARY")
    print("-" * 60)
    for ps in report["pillar_summary"]:
        print(
            f"  Pillar {ps['pillar']}: {ps['pillar_name']:30s} | "
            f"Mean: {ps['mean_intactness']:.1%} | "
            f"Range: [{ps['min_intactness']:.1%}, {ps['max_intactness']:.1%}]"
        )

## 7. Download results

In [ ]:
from google.colab import files

# Download JSON
files.download("./output/benchmark_scorecard.json")

# Download CSV
files.download("./output/benchmark_scorecard.csv")

print("\n✓ Scorecard downloaded (JSON + CSV)")

---

## (Optional) Add a custom indicator

Adding any new indicator takes just a few lines — no pipeline changes needed.

In [ ]:
# Example: Add CPLAND (connectivity) as a custom indicator

def extract_cpland(geometry, config):
    """
    Your CPLAND extraction logic here.
    Could call Darukaa's internal API, compute from FRAGSTATS, etc.
    """
    # Placeholder — replace with your real implementation
    return {"value": 0.2135, "pixels": None}

registry.register(
    name="cpland",
    display_name="Landscape Connectivity (CPLAND)",
    source_type="api",
    extract_fn=extract_cpland,
    unit="%",
    value_range=(0.0, 100.0),
    citation="McGarigal & Marks (1995). FRAGSTATS. USDA Forest Service.",
    tier2_eligible=True,
    pillar=1,
)

print(f"✓ Registry now has {len(registry)} indicators: {registry.names()}")